# Day 1 — Exploratory Data Analysis
**Goal:** Understand the dataset before building anything. Every model decision you make later should be traceable to something you saw here.

Run cells top-to-bottom. All outputs will be saved to `results/eda/`.

In [ ]:
import sys, os
sys.path.insert(0, '..')   # so we can import src/

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
from pathlib import Path
from wordcloud import WordCloud

from src.data_utils import load_dataset

# ── Config ────────────────────────────────────────────────────────────────────
DATASET_PATH = '../data/processed/dataset_v1.jsonl'
RESULTS_DIR  = Path('../results/eda')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

print('Libraries loaded ✓')

## 1. Load and preview the dataset

In [ ]:
df, samples = load_dataset(DATASET_PATH)

print('\n── Shape:', df.shape)
print('── Columns:', df.columns.tolist())
print('── Label distribution:')
print(df['label'].value_counts().rename({1: 'phishing', 0: 'legit'}))

df.head(3)

In [ ]:
# Sanity checks
print('Null values per column:')
print(df.isnull().sum())
print('\nSource distribution:')
print(df['source'].value_counts())

## 2. Label Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
counts = df['label'].value_counts().sort_index()
bars = ax.bar(['Legitimate (0)', 'Phishing (1)'], counts.values,
              color=['#4A9EBF', '#E05C5C'], edgecolor='white', width=0.5)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{val:,}', ha='center', va='bottom', fontweight='bold')
ax.set_title('Label Distribution', fontweight='bold')
ax.set_ylabel('Count')
ax.set_ylim(0, counts.max() * 1.15)
sns.despine()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'label_distribution.png')
plt.show()
print('INSIGHT: Dataset is balanced — good. No class weighting needed for baseline.')

## 3. Email Length Analysis

In [ ]:
df['body_word_count']    = df['body'].str.split().str.len().fillna(0).astype(int)
df['subject_word_count'] = df['subject'].str.split().str.len().fillna(0).astype(int)
df['total_chars']        = df['body'].str.len().fillna(0).astype(int)

print('── Body word count stats by label:')
print(df.groupby('label')['body_word_count'].describe().round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col, title in zip(axes,
    ['body_word_count', 'subject_word_count'],
    ['Body Word Count', 'Subject Word Count']):

    for label, color, name in [(0, '#4A9EBF', 'Legit'), (1, '#E05C5C', 'Phishing')]:
        data = df[df['label']==label][col].clip(upper=500 if 'body' in col else 30)
        ax.hist(data, bins=40, alpha=0.6, color=color, label=name, edgecolor='white')

    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Word count')
    ax.set_ylabel('Frequency')
    ax.legend()
    sns.despine(ax=ax)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'length_distributions.png')
plt.show()
print('INSIGHT: Note differences in body length between classes — this is a usable feature.')

## 4. Top Words — Phishing vs Legitimate

In [ ]:
import re
from collections import Counter

STOPWORDS = set([
    'the','a','an','and','or','but','in','on','at','to','for','of','with',
    'is','was','are','were','be','been','has','have','had','this','that',
    'it','its','by','from','as','i','you','we','he','she','they','your',
    'our','my','will','can','do','not','no','if','so','up','out','about',
    'all','would','could','should','may','just','more','than','also','very',
    'any','other','new','like','get','re','subject','email','mail'
])

def top_words(texts, n=50):
    tokens = []
    for t in texts:
        words = re.findall(r'\b[a-zA-Z]{3,}\b', str(t).lower())
        tokens.extend([w for w in words if w not in STOPWORDS])
    return Counter(tokens).most_common(n)

phishing_words = top_words(df[df['label']==1]['body'])
legit_words    = top_words(df[df['label']==0]['body'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, words, color, title in [
    (axes[0], phishing_words[:20], '#E05C5C', 'Top 20 Words — Phishing'),
    (axes[1], legit_words[:20],    '#4A9EBF', 'Top 20 Words — Legitimate'),
]:
    labels_w, counts = zip(*words)
    ax.barh(list(reversed(labels_w)), list(reversed(counts)), color=color, alpha=0.8)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Frequency')
    sns.despine(ax=ax)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'top_words.png')
plt.show()

## 5. Word Clouds

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, label, color, title in [
    (axes[0], 1, '#E05C5C', 'Phishing Emails'),
    (axes[1], 0, '#4A9EBF', 'Legitimate Emails'),
]:
    text = " ".join(df[df['label']==label]['body'].fillna('').tolist())
    wc = WordCloud(
        width=600, height=350,
        background_color='white',
        colormap='Reds' if label==1 else 'Blues',
        stopwords=STOPWORDS,
        max_words=100,
    ).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(title, fontweight='bold', fontsize=13)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'wordclouds.png', bbox_inches='tight')
plt.show()

## 6. Feature Signal Analysis
These hand-crafted features will be used in the Day 2 baseline classifier.
Visualising them now lets you know which ones will matter.

In [ ]:
URGENCY_WORDS = [
    'urgent', 'immediately', 'action required', 'verify', 'suspended',
    'click here', 'confirm', 'account', 'password', 'winner', 'prize',
    'free', 'limited time', 'expires', 'warning', 'alert'
]

def count_urgency(text):
    t = str(text).lower()
    return sum(t.count(w) for w in URGENCY_WORDS)

def count_links(text):
    return len(re.findall(r'https?://', str(text)))

def caps_ratio(text):
    t = str(text)
    letters = [c for c in t if c.isalpha()]
    if not letters: return 0.0
    return sum(1 for c in letters if c.isupper()) / len(letters)

def exclamation_count(text):
    return str(text).count('!')

df['urgency_count']    = df['body'].apply(count_urgency)
df['link_count']       = df['body'].apply(count_links)
df['caps_ratio']       = df['body'].apply(caps_ratio)
df['exclamation_count']= df['body'].apply(exclamation_count)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
features = ['urgency_count', 'link_count', 'caps_ratio', 'exclamation_count']
titles   = ['Urgency Word Count', 'Link Count', 'CAPS Ratio', 'Exclamation Marks']
clips    = [20, 10, 0.5, 15]

for ax, feat, title, clip in zip(axes.flatten(), features, titles, clips):
    for label, color, name in [(0, '#4A9EBF', 'Legit'), (1, '#E05C5C', 'Phishing')]:
        data = df[df['label']==label][feat].clip(upper=clip)
        ax.hist(data, bins=30, alpha=0.6, color=color, label=name, edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.legend()
    sns.despine(ax=ax)

plt.suptitle('Feature Distributions: Phishing vs Legitimate', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'feature_distributions.png', bbox_inches='tight')
plt.show()
print('INSIGHT: Features that show clear separation between classes = strong baseline predictors.')

## 7. Correlation heatmap of engineered features

In [ ]:
feature_cols = ['body_word_count','subject_word_count','urgency_count',
                'link_count','caps_ratio','exclamation_count','label']
corr = df[feature_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'correlation_heatmap.png')
plt.show()
print('INSIGHT: Features highly correlated with label (last row) are most predictive.')

## 8. EDA Summary
Run this cell to print a concise summary to paste into your README.

In [ ]:
print('═' * 55)
print('  EDA SUMMARY — paste into README / results.md')
print('═' * 55)
print(f'Total samples   : {len(df):,}')
print(f'Phishing        : {df["label"].sum():,}')
print(f'Legitimate      : {(df["label"]==0).sum():,}')
print(f'Avg body length (phishing): {df[df["label"]==1]["body_word_count"].mean():.0f} words')
print(f'Avg body length (legit)   : {df[df["label"]==0]["body_word_count"].mean():.0f} words')
print(f'Avg links (phishing)      : {df[df["label"]==1]["link_count"].mean():.2f}')
print(f'Avg links (legit)         : {df[df["label"]==0]["link_count"].mean():.2f}')
print(f'Avg urgency words (phishing): {df[df["label"]==1]["urgency_count"].mean():.2f}')
print(f'Avg urgency words (legit)   : {df[df["label"]==0]["urgency_count"].mean():.2f}')
print('═' * 55)
print(f'\nAll EDA charts saved to: results/eda/')